In [ ]:
import time
%matplotlib qt
import numpy as np
import pandas as pd
import seaborn as sns
import nibabel as nib
import matplotlib.pyplot as plt


def get_roi_values(img_path, mask_path):
    """Loads a NIfTI volume, isolates voxel arrays inside a mask region, and calculates the 10th percentile."""
    img = nib.load(img_path)
    mask = nib.load(mask_path)

    img_data = img.get_fdata()
    mask_data = mask.get_fdata()

    # Extract raw HU values where the target mask segmentation is active
    roi_values = img_data[mask_data > 0]

    # Calculate the 10th percentile metric safely
    p10 = np.percentile(roi_values, 10) if len(roi_values) > 0 else np.nan
    return roi_values, p10

In [ ]:
# 1. Path configurations (Standardized 'Myel' to 'MM')
data_config = [
    {
        "img": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\healthy_010_konv.nii.gz",
        "mask": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\healthy_010_spine_seg.nii.gz",
        "label": "Healthy"
    },
    {
        "img": r"E:\DATA_Myelomy\Myel_001\myel_001_konv.nii.gz",
        "mask": r"E:\DATA_Myelomy\Myel_001\myel_001_spine_seg.nii.gz",
        "label": "MM"
    }
]

extracted_rows = []

# ==============================================================================
# 2. DATA EXTRACTION & DATAFRAME CONSTRUCTION
# ==============================================================================
for profile in data_config:
    hu_values = get_roi_values(profile['img'], profile['mask'])

    # Create an explicit temporary DataFrame per subject profile
    temp_df = pd.DataFrame({
        'HU': hu_values,
        'Group': profile['label']
    })
    extracted_rows.append(temp_df)

# Combine into a single master long-format DataFrame for Seaborn
plot_df = pd.concat(extracted_rows, ignore_index=True)

# ==============================================================================
# 3. HIGH-DENSITY VISUALIZATION PIPELINE (Side-by-Side Analysis)
# ==============================================================================
sns.set_theme(style="darkgrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Explicit cohort ordering and palette definitions
group_order = ["Healthy", "MM"]
color_palette = {"Healthy": "#66c2a5", "MM": "#fc8d62"}

# --------------------------------------------------------------------------
# SUBPLOT 1: Boxplot (Quantile Range Distribution Profile)
# --------------------------------------------------------------------------
sns.boxplot(
    data=plot_df,
    x='Group',
    y='HU',
    order=group_order,
    hue='Group',
    palette=color_palette,
    showfliers=False,
    legend=False,
    ax=ax1
)
ax1.set_title('Boxplot: HU Intensity Distribution', fontsize=14, pad=12, fontweight='bold')
ax1.set_xlabel('Clinical Group', fontsize=11)
ax1.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# --------------------------------------------------------------------------
# SUBPLOT 2: Violin Plot (Probability Density Profile)
# --------------------------------------------------------------------------
sns.violinplot(
    data=plot_df,
    x='Group',
    y='HU',
    order=group_order,
    hue='Group',
    palette=color_palette,
    inner="quartile",
    cut=0,
    legend=False,
    ax=ax2
)
ax2.set_title('Violin Plot: Density and Probability Structure', fontsize=14, pad=12, fontweight='bold')
ax2.set_xlabel('Clinical Group', fontsize=11)
ax2.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Supertitle and formatting configurations in English
plt.suptitle('Vertebrae ROI Density Voxel Analysis: Healthy vs Multiple Myeloma (MM)', fontsize=16, y=0.98, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Path configurations (Standardized text labels to English)
data_config = [
    {
        "img": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\healthy_010_konv.nii.gz",
        "mask": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\liver_segment_8.nii.gz",
        "label": "Healthy 10"
    },
    {
        "img": r"E:\DATA_Healthy_and_Myelom\DATA_MM_Stage1\Myel_001\myel_001_konv.nii.gz",
        "mask": r"E:\DATA_Healthy_and_Myelom\DATA_MM_Stage1\Myel_001\liver_segment_8.nii.gz",
        "label": "MM Stage 1"
    }
]

extracted_records = []
p10_metrics_map = {}

# ==============================================================================
# 2. FILE EXTRACTION & ANALYSIS
# ==============================================================================
for profile in data_config:
    voxel_intensities, computed_p10 = get_roi_values(profile['img'], profile['mask'])

    # Store calculated statistic for custom graph overlays
    p10_metrics_map[profile['label']] = computed_p10
    print(f"[{profile['label']}] 10th Percentile Threshold: {computed_p10:.2f} HU")

    # Group measurements inside a structural data frame block
    temp_df = pd.DataFrame({
        'HU': voxel_intensities,
        'Cohort': profile['label']
    })
    extracted_records.append(temp_df)

# Assemble individual components into a single primary plotting frame
plot_df = pd.concat(extracted_records, ignore_index=True)

# ==============================================================================
# 3. GRAPHICAL GRID PIPELINE GENERATION
# ==============================================================================
sns.set_theme(style="darkgrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7.5))

cohort_order = [p['label'] for p in data_config]
palette_colors = {"Healthy 10": "#66c2a5", "MM Stage 1": "#fc8d62"}

# --------------------------------------------------------------------------
# SUBPLOT 1: Boxplot (Quantile Range Outlines)
# --------------------------------------------------------------------------
sns.boxplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    showfliers=False,
    legend=False,
    ax=ax1
)
ax1.set_title('Boxplot: HU Distribution Range', fontsize=13, pad=12, fontweight='bold')
ax1.set_xlabel('Patient Cohort Case', fontsize=11)
ax1.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Superimpose specific 10th percentile horizontal markers per case column
for class_idx, class_label in enumerate(cohort_order):
    stat_val = p10_metrics_map[class_label]
    if not np.isnan(stat_val):
        ax1.axhline(
            y=stat_val,
            color=palette_colors[class_label],
            linestyle=':',
            linewidth=2,
            alpha=0.9,
            label=f"{class_label} 10th Pctl ({stat_val:.1f})"
        )
ax1.legend(title="Statistical Thresholds", loc="lower left")

# --------------------------------------------------------------------------
# SUBPLOT 2: Violin Plot (Probability Density Outlines)
# --------------------------------------------------------------------------
sns.violinplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    inner="quartile",
    cut=0,
    legend=False,
    ax=ax2
)
ax2.set_title('Violin Plot: Continuous Structural Density Profile', fontsize=13, pad=12, fontweight='bold')
ax2.set_xlabel('Patient Cohort Case', fontsize=11)
ax2.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Add a subtle baseline context reference at 0 HU (Water density benchmark)
for ax in (ax1, ax2):
    ax.axhline(0, color='#2b2b2b', linestyle='--', linewidth=1, alpha=0.4)

# Formulate titles and layouts in English
plt.suptitle('Quantitative Analysis of Radiomic Liver Segment 8 Voxel Intensity Profiles', fontsize=15, y=0.97, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Path configurations (Standardized text labels for clinical accuracy)
data_config = [
    {
        "img": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\healthy_010_konv.nii.gz",
        "mask": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\healthy_010_konv_fat_segment.nii.gz",
        "label": "Healthy 10"
    },
    {
        "img": r"E:\DATA_Healthy_and_Myelom\DATA_MM_Stage1\Myel_003\myel_003_konv.nii.gz",
        "mask": r"E:\DATA_Healthy_and_Myelom\DATA_MM_Stage1\Myel_003\myel_003_konv_fat_segment.nii.gz",
        "label": "MM Stage 1 (Pt 3)"
    }
]

extracted_records = []
p10_metrics_map = {}

# ==============================================================================
# 2. DATA LOAD & CALCULATIONS LOOP
# ==============================================================================
for profile in data_config:
    voxel_intensities, computed_p10 = get_roi_values(profile['img'], profile['mask'])

    # Store calculated statistic for personalized chart indicator lines
    p10_metrics_map[profile['label']] = computed_p10
    print(f"[{profile['label']}] 10th Percentile Threshold: {computed_p10:.2f} HU")

    # Structure data to build a long-format DataFrame
    temp_df = pd.DataFrame({
        'HU': voxel_intensities,
        'Cohort': profile['label']
    })
    extracted_records.append(temp_df)

# Merge rows cleanly into a unified master frame
plot_df = pd.concat(extracted_records, ignore_index=True)

# ==============================================================================
# 3. SIDE-BY-SIDE GRAPH PLOTTING PIPELINE
# ==============================================================================
sns.set_theme(style="darkgrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7.5))

cohort_order = [p['label'] for p in data_config]
palette_colors = {"Healthy 10": "#66c2a5", "MM Stage 1 (Pt 3)": "#fc8d62"}

# --------------------------------------------------------------------------
# SUBPLOT 1: Boxplot (Quantile Range Range Distribution Profile)
# --------------------------------------------------------------------------
sns.boxplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    showfliers=False,
    legend=False,
    ax=ax1
)
ax1.set_title('Boxplot: HU Distribution (FAT ROI)', fontsize=13, pad=12, fontweight='bold')
ax1.set_xlabel('Patient Group / Case', fontsize=11)
ax1.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Superimpose the actual computed 10th percentile horizontal markers
for class_label in cohort_order:
    stat_val = p10_metrics_map[class_label]
    if not np.isnan(stat_val):
        ax1.axhline(
            y=stat_val,
            color=palette_colors[class_label],
            linestyle=':',
            linewidth=2,
            alpha=0.9,
            label=f"{class_label} 10th Pctl ({stat_val:.1f} HU)"
        )
ax1.legend(title="Statistical Thresholds", loc="lower left")

# --------------------------------------------------------------------------
# SUBPLOT 2: Violin Plot (Continuous Density Distribution Profile)
# --------------------------------------------------------------------------
sns.violinplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    inner="quartile",
    cut=0,
    legend=False,
    ax=ax2
)
ax2.set_title('Violin Plot: Density Profile (FAT ROI)', fontsize=13, pad=12, fontweight='bold')
ax2.set_xlabel('Patient Group / Case', fontsize=11)
ax2.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Add a standard dashed context reference line at 0 HU (Water density reference)
for ax in (ax1, ax2):
    ax.axhline(0, color='#2b2b2b', linestyle='--', linewidth=1, alpha=0.4)

# Formulate headers, titles and adjust layout dimensions completely in English
plt.suptitle('Hounsfield Unit (HU) Distribution Analysis within Adipose Tissue (FAT ROI)', fontsize=15, y=0.97, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Path configurations (Standardized text labels for cohort uniformity)
data_config = [
    {
        "img": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\healthy_010_konv.nii.gz",
        "mask": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_010\healthy_010_konv_muscle_segment.nii.gz",
        "label": "Healthy 10"
    },
    {
        "img": r"E:\DATA_Healthy_and_Myelom\DATA_MM_Stage1\Myel_003\myel_003_konv.nii.gz",
        "mask": r"E:\DATA_Healthy_and_Myelom\DATA_MM_Stage1\Myel_003\myel_003_konv_muscle_segment.nii.gz",
        "label": "MM Stage 1 (Pt 3)"
    }
]

extracted_records = []
p10_metrics_map = {}

# ==============================================================================
# 2. DATA LOAD & CALCULATIONS LOOP
# ==============================================================================
for profile in data_config:
    voxel_intensities, computed_p10 = get_roi_values(profile['img'], profile['mask'])

    # Store calculated statistic for personalized chart indicator lines
    p10_metrics_map[profile['label']] = computed_p10
    print(f"[{profile['label']}] 10th Percentile Threshold: {computed_p10:.2f} HU")

    # Structure data to build a long-format DataFrame
    temp_df = pd.DataFrame({
        'HU': voxel_intensities,
        'Cohort': profile['label']
    })
    extracted_records.append(temp_df)

# Merge individual frames cleanly into a unified master frame
plot_df = pd.concat(extracted_records, ignore_index=True)

# ==============================================================================
# 3. SIDE-BY-SIDE GRAPH PLOTTING PIPELINE
# ==============================================================================
sns.set_theme(style="darkgrid")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7.5))

cohort_order = [p['label'] for p in data_config]
palette_colors = {"Healthy 10": "#66c2a5", "MM Stage 1 (Pt 3)": "#fc8d62"}

# --------------------------------------------------------------------------
# SUBPLOT 1: Boxplot (Quantile Range Range Distribution Profile)
# --------------------------------------------------------------------------
sns.boxplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    showfliers=False,
    legend=False,
    ax=ax1
)
ax1.set_title('Boxplot: HU Distribution (MUSCLE ROI)', fontsize=13, pad=12, fontweight='bold')
ax1.set_xlabel('Patient Group / Case', fontsize=11)
ax1.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Superimpose the actual computed 10th percentile horizontal markers
for class_label in cohort_order:
    stat_val = p10_metrics_map[class_label]
    if not np.isnan(stat_val):
        ax1.axhline(
            y=stat_val,
            color=palette_colors[class_label],
            linestyle=':',
            linewidth=2,
            alpha=0.9,
            label=f"{class_label} 10th Pctl ({stat_val:.1f} HU)"
        )
ax1.legend(title="Statistical Thresholds", loc="lower left")

# --------------------------------------------------------------------------
# SUBPLOT 2: Violin Plot (Continuous Density Distribution Profile)
# --------------------------------------------------------------------------
sns.violinplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    inner="quartile",
    cut=0,
    legend=False,
    ax=ax2
)
ax2.set_title('Violin Plot: Density Profile (MUSCLE ROI)', fontsize=13, pad=12, fontweight='bold')
ax2.set_xlabel('Patient Group / Case', fontsize=11)
ax2.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Add a standard dashed context reference line at 0 HU (Water density benchmark)
for ax in (ax1, ax2):
    ax.axhline(0, color='#2b2b2b', linestyle='--', linewidth=1, alpha=0.4)

# Set headers, titles, and layout spaces fully in English
plt.suptitle('Hounsfield Unit (HU) Distribution Analysis within Muscle Tissue (MUSCLE ROI)', fontsize=15, y=0.97, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def visualize_dual_sync(p1_cfg, p2_cfg):
    """
    Loads two 3D NIfTI volumes, runs an optimized dual synchronized slice animation
    with overlaid bounding box patches, and returns flattened voxel intensity arrays.
    """
    # 1. Load 3D volume arrays into memory
    img1 = nib.load(p1_cfg['img']).get_fdata()
    img2 = nib.load(p2_cfg['img']).get_fdata()

    # Initialize a dual interactive plotting window layout
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 6.5))

    # Standard Bone structural window leveling profile for CT visualization
    vmin, vmax = -200, 800

    # Determine execution duration bounded by the shortest overlapping target range
    z_len1 = p1_cfg['z_range'][1] - p1_cfg['z_range'][0]
    z_len2 = p2_cfg['z_range'][1] - p2_cfg['z_range'][0]
    num_frames = min(z_len1, z_len2)

    # Calculate initial static slice values to set base frame states
    init_z1 = p1_cfg['z_range'][0]
    init_z2 = p2_cfg['z_range'][0]

    # --------------------------------------------------------------------------
    # HIGH-PERFORMANCE INITIALIZATION (Define Artists Once Outside the Loop)
    # --------------------------------------------------------------------------
    # Subplot 1: Patient Cohort 1 Setup
    im1 = ax1.imshow(img1[:, :, init_z1].T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
    rect1 = plt.Rectangle(
        (p1_cfg['x_range'][0], p1_cfg['y_range'][0]),
        p1_cfg['x_range'][1] - p1_cfg['x_range'][0],
        p1_cfg['y_range'][1] - p1_cfg['y_range'][0],
        linewidth=2, edgecolor='#66c2a5', facecolor='none'
    )
    ax1.add_patch(rect1)
    ax1.axis('on')

    # Subplot 2: Patient Cohort 2 Setup
    im2 = ax2.imshow(img2[:, :, init_z2].T, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
    rect2 = plt.Rectangle(
        (p2_cfg['x_range'][0], p2_cfg['y_range'][0]),
        p2_cfg['x_range'][1] - p2_cfg['x_range'][0],
        p2_cfg['y_range'][1] - p2_cfg['y_range'][0],
        linewidth=2, edgecolor='#fc8d62', facecolor='none'
    )
    ax2.add_patch(rect2)
    ax2.axis('on')

    plt.show(block=False)
    fig.canvas.draw()

    # --------------------------------------------------------------------------
    # LIGHTWEIGHT RE-DRAW LOOP (Updates only the underlying array state data)
    # --------------------------------------------------------------------------
    for i in range(num_frames):
        z1 = p1_cfg['z_range'][0] + i
        z2 = p2_cfg['z_range'][0] + i

        # Update underlying image matrix arrays smoothly without recreating elements
        im1.set_data(img1[:, :, z1].T)
        im2.set_data(img2[:, :, z2].T)

        # Update title text elements dynamically
        ax1.set_title(f"{p1_cfg['label']} | Slice ID: {z1}", fontsize=11)
        ax2.set_title(f"{p2_cfg['label']} | Slice ID: {z2}", fontsize=11)

        # Force interactive frame updates
        fig.canvas.blit(ax1.bbox)
        fig.canvas.blit(ax2.bbox)
        fig.canvas.flush_events()
        time.sleep(0.04)

    plt.close(fig)

    # --------------------------------------------------------------------------
    # QUANTITATIVE DATA EXTRACTION & EXTRACTION
    # --------------------------------------------------------------------------
    vals1 = img1[
        p1_cfg['x_range'][0]:p1_cfg['x_range'][1],
        p1_cfg['y_range'][0]:p1_cfg['y_range'][1],
        p1_cfg['z_range'][0]:p1_cfg['z_range'][1]
    ].flatten()
    p10_v1 = np.percentile(vals1, 10)
    print(f"[{p1_cfg['label']}] 10th Percentile Threshold: {p10_v1:.2f} HU")

    vals2 = img2[
        p2_cfg['x_range'][0]:p2_cfg['x_range'][1],
        p2_cfg['y_range'][0]:p2_cfg['y_range'][1],
        p2_cfg['z_range'][0]:p2_cfg['z_range'][1]
    ].flatten()
    p10_v2 = np.percentile(vals2, 10)
    print(f"[{p2_cfg['label']}] 10th Percentile Threshold: {p10_v2:.2f} HU")

    return vals1, vals2, p10_v1, p10_v2


# ==============================================================================
# PIPELINE CONFIGURATION & RUNTIME EXECUTION
# ==============================================================================
p1_config = {
    "img": r"E:\DATA_Healthy_and_Myelom\DATA_Healthy\Healthy_001\healthy_001_konv.nii.gz",
    "label": "Healthy 1",
    "x_range": (100, 200), "y_range": (400, 500), "z_range": (100, 300)
}

p2_config = {
    "img": r"E:\DATA_Myelomy\Myel_001\myel_001_konv.nii.gz",
    "label": "MM Stage 1",
    "x_range": (100, 300), "y_range": (460, 500), "z_range": (550, 750)
}

# Run optimized video slicing matrix animator
v1_data, v2_data, p10_1, p10_2 = visualize_dual_sync(p1_config, p2_config)

# ==============================================================================
# POST-PROCESSING DENSITY STATISTICAL ANALYSIS VIZ
# ==============================================================================
# Re-route rendering engine seamlessly back to inline notebook display
# %matplotlib inline

sns.set_theme(style="darkgrid")

# Restructure extracted flattened metrics into a standardized long-format DataFrame
df_v1 = pd.DataFrame({'HU': v1_data, 'Cohort': p1_config['label']})
df_v2 = pd.DataFrame({'HU': v2_data, 'Cohort': p2_config['label']})
plot_df = pd.concat([df_v1, df_v2], ignore_index=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7.5))
cohort_order = [p1_config['label'], p2_config['label']]
palette_colors = {p1_config['label']: "#66c2a5", p2_config['label']: "#fc8d62"}
p10_map = {p1_config['label']: p10_1, p2_config['label']: p10_2}

# --------------------------------------------------------------------------
# SUBPLOT 1: Boxplot (Quantile Structural Range Profiling)
# --------------------------------------------------------------------------
sns.boxplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    showfliers=False,
    legend=False,
    ax=ax1
)
ax1.set_title('Boxplot: HU Background Distribution Ranges', fontsize=13, pad=12, fontweight='bold')
ax1.set_xlabel('Patient Study Cohorts', fontsize=11)
ax1.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Dynamically overlay specific calculated 10th percentile indicator reference lines
for target_cohort in cohort_order:
    stat_val = p10_map[target_cohort]
    ax1.axhline(
        y=stat_val,
        color=palette_colors[target_cohort],
        linestyle=':',
        linewidth=2.5,
        alpha=0.9,
        label=f"{target_cohort} 10th Pctl ({stat_val:.1f} HU)"
    )
ax1.legend(title="Calculated Metrics", loc="lower left")

# --------------------------------------------------------------------------
# SUBPLOT 2: Violin Plot (Continuous Voxel Density Structural Probability)
# --------------------------------------------------------------------------
sns.violinplot(
    data=plot_df,
    x='Cohort',
    y='HU',
    order=cohort_order,
    hue='Cohort',
    palette=palette_colors,
    inner="quartile",
    cut=0,
    legend=False,
    ax=ax2
)
ax2.set_title('Violin Plot: Density and Probability Profiles', fontsize=13, pad=12, fontweight='bold')
ax2.set_xlabel('Patient Study Cohorts', fontsize=11)
ax2.set_ylabel('Hounsfield Units (HU)', fontsize=11)

# Superimpose a universal baseline zero water marker line across both axes
for ax in (ax1, ax2):
    ax.axhline(0, color='#2b2b2b', linestyle='--', linewidth=1, alpha=0.4)

# Global canvas packaging in clean English
plt.suptitle(
    f"Quantitative Evaluation of Background ROI Structural Densities:\n"
    f"{p1_config['label']} vs {p2_config['label']}",
    fontsize=15, y=0.96, fontweight='bold'
)
plt.tight_layout()
plt.show()